# Blue-Green, Canary, and SLO-Based Rollouts
Companion notebook for the **Blue-Green, Canary, and SLO-Based Rollouts** section of Chapter 12.

Don't gate a rollout only on "service is up" -- tie each step to SLO-based checks defined *before* release work starts. This notebook loads the real `rollout.yaml` shipped alongside it, the book's example unchanged.

In [1]:
print(open("rollout.yaml").read())

rollout:
  strategy: canary
  steps:
    - traffic: 5
      hold_minutes: 15
    - traffic: 20
      hold_minutes: 30
    - traffic: 50
      hold_minutes: 30
  gates:
    p95_latency_ms: "<= 2500"
    error_rate: "<= 1%"
    task_success: ">= 92%"
  rollback:
    on_gate_fail: true
    target_version: blue



In [2]:
import yaml

rollout = yaml.safe_load(open("rollout.yaml"))["rollout"]
print(rollout)

{'strategy': 'canary', 'steps': [{'traffic': 5, 'hold_minutes': 15}, {'traffic': 20, 'hold_minutes': 30}, {'traffic': 50, 'hold_minutes': 30}], 'gates': {'p95_latency_ms': '<= 2500', 'error_rate': '<= 1%', 'task_success': '>= 92%'}, 'rollback': {'on_gate_fail': True, 'target_version': 'blue'}}


The book shows the config but not the gate evaluator -- a small one, parsing the `"<= 2500"` / `">= 92%"` style conditions and checking each canary step's metrics against them:

In [3]:
def parse_gate(condition: str):
    condition = condition.strip()
    if condition.startswith("<="):
        return "<=", float(condition[2:].strip().rstrip("%"))
    if condition.startswith(">="):
        return ">=", float(condition[2:].strip().rstrip("%"))
    raise ValueError(f"unsupported gate condition: {condition}")


def check_gates(gates: dict, metrics: dict) -> list[str]:
    failures = []
    for name, condition in gates.items():
        op, threshold = parse_gate(condition)
        value = metrics[name]
        ok = value <= threshold if op == "<=" else value >= threshold
        if not ok:
            failures.append(f"{name}={value} fails '{condition}'")
    return failures

A canary runner that walks the traffic steps in order, stopping (and rolling back) at the first step whose metrics fail a gate:

In [4]:
def run_canary(rollout: dict, metrics_by_step: list[dict]) -> dict:
    for step, metrics in zip(rollout["steps"], metrics_by_step):
        failures = check_gates(rollout["gates"], metrics)
        if failures:
            return {
                "status": "rolled_back",
                "failed_at_traffic": step["traffic"],
                "reasons": failures,
                "target_version": rollout["rollback"]["target_version"],
            }
    return {"status": "fully_rolled_out"}

In [5]:
healthy_metrics = [
    {"p95_latency_ms": 2100, "error_rate": 0.4, "task_success": 94},
    {"p95_latency_ms": 2200, "error_rate": 0.5, "task_success": 93},
    {"p95_latency_ms": 2300, "error_rate": 0.6, "task_success": 93},
]
print("all steps healthy:", run_canary(rollout, healthy_metrics))

bad_metrics = [
    {"p95_latency_ms": 2100, "error_rate": 0.4, "task_success": 94},   # 5% step: fine
    {"p95_latency_ms": 2800, "error_rate": 2.1, "task_success": 88},   # 20% step: fails every gate
]
print("20% step degrades:  ", run_canary(rollout, bad_metrics))

all steps healthy: {'status': 'fully_rolled_out'}
20% step degrades:   {'status': 'rolled_back', 'failed_at_traffic': 20, 'reasons': ["p95_latency_ms=2800 fails '<= 2500'", "error_rate=2.1 fails '<= 1%'", "task_success=88 fails '>= 92%'"], 'target_version': 'blue'}


Notice `task_success` is one of the three gates, right alongside latency and error rate -- exactly the point the book makes: a release can keep the API healthy (low error rate, fine latency) while still making the underlying workflow worse. Gating on uptime alone would have let the second rollout through.

**Blue-green** is the simpler sibling of this pattern: two full versions ready, one switch of traffic, one switch back if something goes wrong -- no staged traffic percentages, just a clean cutover. Use it when you want a simple rollback path more than you want gradual exposure.